# Apache Hudi - Structured Streaming with Hudi

Apache Hudi can be used as a Structured Streaming sink, allowing Spark streaming jobs to continuously write inserts and upserts into a Hudi table.
This is a common pattern for near-real-time data lake ingestion where events arrive continuously and the target table must stay queryable while new data is being written.

## What you will learn

In this notebook, you will learn:

- Creating a Spark Structured Streaming source using `rate`
- Transforming streaming rows into Hudi records
- Writing streaming data into a Hudi Copy-On-Write (COW) table
- Configuring checkpointing for streaming fault tolerance
- Running a short streaming query with a processing-time trigger
- Stopping the streaming query safely
- Reading and verifying the generated Hudi table

## Step 1 — Create Spark session

This notebook assumes that the Hudi bundle JAR is already available in the Spark Docker image under `/opt/spark/jars`.

Because of that, we do not use `spark.jars.packages` here. This avoids Ivy/Maven downloads from inside the notebook.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Hudi-09-Structured-Streaming")
    .master("spark://spark-master:7077")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.2


## Step 2 — Define shared paths and table name

This notebook is independent and creates its own Hudi table and checkpoint folder.

The checkpoint path must be stable for a real streaming job. For a tutorial run, we clean it before starting.

In [2]:
BASE_PATH = "/workspace/data/hudi"
CHECKPOINT_BASE_PATH = "/workspace/data/checkpoints"

TABLE_NAME = "riders_structured_streaming"
TABLE_PATH = f"{BASE_PATH}/{TABLE_NAME}"
CHECKPOINT_PATH = f"{CHECKPOINT_BASE_PATH}/{TABLE_NAME}"

print("Table name:", TABLE_NAME)
print("Table path:", TABLE_PATH)
print("Checkpoint path:", CHECKPOINT_PATH)

Table name: riders_structured_streaming
Table path: /workspace/data/hudi/riders_structured_streaming
Checkpoint path: /workspace/data/checkpoints/riders_structured_streaming


## Step 3 — Clean previous run

For this tutorial notebook, we remove the old Hudi table and checkpoint folder.

In production, do not delete the checkpoint folder unless you intentionally want to reset the streaming job state.

In [3]:
import shutil
import os

for path in [TABLE_PATH, CHECKPOINT_PATH]:
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Removed: {path}")
    else:
        print(f"No existing path found: {path}")

No existing path found: /workspace/data/hudi/riders_structured_streaming
No existing path found: /workspace/data/checkpoints/riders_structured_streaming


## Step 4 — Define Hudi streaming write options

The table uses:

- `rider` as the record key
- `city` as the partition path
- `ts` as the precombine field
- `COPY_ON_WRITE` table type

The checkpoint location is used by Spark Structured Streaming, not by Hudi directly.

In [4]:
hudi_streaming_options = {
    "hoodie.table.name": TABLE_NAME,
    "hoodie.datasource.write.table.name": TABLE_NAME,
    "hoodie.datasource.write.recordkey.field": "rider",
    "hoodie.datasource.write.partitionpath.field": "city",
    "hoodie.datasource.write.precombine.field": "ts",
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
    "hoodie.datasource.write.operation": "upsert",
    "checkpointLocation": CHECKPOINT_PATH,
}

## Step 5 — Create a streaming source

The `rate` source generates rows continuously.

It produces two columns:

- `timestamp`
- `value`

We transform these into a simple rider event structure.

In [5]:
from pyspark.sql.functions import col, lit, current_timestamp, concat

stream_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 1)
    .option("numPartitions", 1)
    .load()
)

rider_events = (
    stream_df
    .withColumn("rider", concat(lit("r"), col("value").cast("string")))
    .withColumn("city", lit("SFO"))
    .withColumn("ts", current_timestamp())
    .select("rider", "city", "ts")
)

print("Streaming DataFrame schema:")
rider_events.printSchema()

Streaming DataFrame schema:
root
 |-- rider: string (nullable = true)
 |-- city: string (nullable = false)
 |-- ts: timestamp (nullable = false)



## Step 6 — Start streaming writes into Hudi

This starts a short streaming query. The query writes micro-batches into a Hudi table every 5 seconds.

The query is intentionally not left running forever in this tutorial notebook.

In [6]:
query = (
    rider_events.writeStream
    .format("hudi")
    .options(**hudi_streaming_options)
    .outputMode("append")
    .trigger(processingTime="5 seconds")
    .start(TABLE_PATH)
)

print("Streaming query started.")
print("Query id:", query.id)

26/04/25 19:54:24 WARN ConfigUtils: Could not read properties from /workspace/data/hudi/riders_structured_streaming/.hoodie/hoodie.properties: java.io.FileNotFoundException: File file:/workspace/data/hudi/riders_structured_streaming/.hoodie/hoodie.properties does not exist
26/04/25 19:54:24 WARN ConfigUtils: Could not read properties from /workspace/data/hudi/riders_structured_streaming/.hoodie/hoodie.properties.backup: java.io.FileNotFoundException: File file:/workspace/data/hudi/riders_structured_streaming/.hoodie/hoodie.properties.backup does not exist
26/04/25 19:54:25 WARN ConfigUtils: Could not read properties from /workspace/data/hudi/riders_structured_streaming/.hoodie/hoodie.properties: java.io.FileNotFoundException: File file:/workspace/data/hudi/riders_structured_streaming/.hoodie/hoodie.properties does not exist
26/04/25 19:54:25 WARN ConfigUtils: Could not read properties from /workspace/data/hudi/riders_structured_streaming/.hoodie/hoodie.properties.backup: java.io.FileNo

## Step 7 — Let the stream run briefly

We wait long enough for a few micro-batches to write data.

If your local cluster is slower on the first run, increase the sleep time.

In [7]:
import time

time.sleep(15)

print("Query active:", query.isActive)
print("Recent progress:")
for progress in query.recentProgress:
    print(progress)

26/04/25 19:54:31 WARN HoodieWriteConfig: Embedded timeline server is disabled, fallback to use direct marker type for spark


# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf
# WARNING: Unable to attach Serviceability Agent. You can try again with escalated privileges. Two options: a) use -Djol.tryWithSudo=true to try with sudo; b) echo 0 | sudo tee /proc/sys/kernel/yama/ptrace_scope
Query active: True
Recent progress:


## Step 8 — Stop the streaming query safely

Always stop tutorial streaming queries when you are done.

This avoids leaving a background stream running in the notebook kernel.

In [8]:
if query.isActive:
    query.stop()
    print("Streaming query stopped.")
else:
    print("Streaming query was already stopped.")

26/04/25 19:54:44 WARN DAGScheduler: Failed to cancel job group 7aa84e32-fa7e-4d39-9b85-c35837b4d9e3. Cannot find active jobs for it.
26/04/25 19:54:45 WARN ConfigUtils: Could not read properties from /workspace/data/hudi/riders_structured_streaming/.hoodie/metadata/.hoodie/hoodie.properties: java.nio.channels.ClosedByInterruptException
26/04/25 19:54:45 WARN ConfigUtils: Could not read properties from /workspace/data/hudi/riders_structured_streaming/.hoodie/metadata/.hoodie/hoodie.properties.backup: java.io.FileNotFoundException: File file:/workspace/data/hudi/riders_structured_streaming/.hoodie/metadata/.hoodie/hoodie.properties.backup does not exist
26/04/25 19:54:45 WARN ConfigUtils: Interrupted while waiting


26/04/25 19:54:52 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 22383 milliseconds
26/04/25 19:54:52 WARN DAGScheduler: Failed to cancel job group 7aa84e32-fa7e-4d39-9b85-c35837b4d9e3. Cannot find active jobs for it.
Streaming query stopped.


## Step 9 — Read the generated Hudi table

After the stream stops, we read the Hudi table as a normal batch table.

In [9]:
stream_table_df = spark.read.format("hudi").load(TABLE_PATH)
stream_table_df.createOrReplaceTempView("riders_stream")

spark.sql("""
SELECT
    rider,
    city,
    ts,
    _hoodie_commit_time,
    _hoodie_record_key,
    _hoodie_partition_path
FROM riders_stream
ORDER BY rider
""").show(truncate=False)

+-----+----+---+-------------------+------------------+----------------------+
|rider|city|ts |_hoodie_commit_time|_hoodie_record_key|_hoodie_partition_path|
+-----+----+---+-------------------+------------------+----------------------+
+-----+----+---+-------------------+------------------+----------------------+



## Step 10 — Count records and commits

Each micro-batch can create a Hudi commit. The exact number depends on trigger timing and execution speed.

In [10]:
print("Record count:", stream_table_df.count())

commits = [
    row["_hoodie_commit_time"]
    for row in stream_table_df
        .select("_hoodie_commit_time")
        .distinct()
        .orderBy("_hoodie_commit_time")
        .collect()
]

print("Commit instants visible in the table:")
for commit in commits:
    print(commit)

Record count: 0
Commit instants visible in the table:


## Step 11 — Inspect checkpoint and table files

The checkpoint folder stores Structured Streaming progress.

The Hudi table path stores table data, partitions, and Hudi metadata.

In [11]:
from pathlib import Path

def print_files(root_path, title, max_files=80):
    root = Path(root_path)
    print(title)
    if not root.exists():
        print("Path does not exist.")
        return

    files = [p for p in sorted(root.rglob("*")) if p.is_file()]
    for path in files[:max_files]:
        print(path.relative_to(root))
    if len(files) > max_files:
        print(f"... {len(files) - max_files} more files")

print_files(TABLE_PATH, "Hudi table files:")
print()
print_files(CHECKPOINT_PATH, "Structured Streaming checkpoint files:")

Hudi table files:
.hoodie/.hoodie.properties.crc
.hoodie/.index_defs/.index.json.crc
.hoodie/.index_defs/index.json
.hoodie/hoodie.properties
.hoodie/metadata/.hoodie/.hoodie.properties.crc
.hoodie/metadata/.hoodie/hoodie.properties
.hoodie/metadata/.hoodie/timeline/.00000000000000000.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000000.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000000_20260425195436341.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001_20260425195438769.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002_20260425195442876.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.

## Step 12 — Summary

In this notebook, you wrote streaming data into a Hudi table using Spark Structured Streaming.

Key takeaways:

- Hudi can be used as a streaming sink.
- Spark checkpoints are required for reliable streaming progress tracking.
- The Hudi record key and precombine field are still important in streaming upserts.
- Tutorial streaming queries should be stopped explicitly.
- After streaming writes complete, the Hudi table can be queried using normal batch reads.